# Notebook 03 - Feature Engineering
## Fake News Detection - NLP Assignment
### Person 2: W.A. Irusha Madushan (CIT-24-01-0514)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# Load the cleaned dataset
df = pd.read_csv('../data/cleaned_data.csv')

# Drop any remaining null values in processed_text
df = df.dropna(subset=['processed_text'])

print("Dataset loaded successfully")
print("Shape:", df.shape)

Dataset loaded successfully
Shape: (72041, 5)


## 1. Train/Test Split

Before creating features, the dataset is split into training and 
testing sets. I used an 80/20 split which means 80% of the data 
is used for training and 20% is used for testing. This is a 
standard approach in machine learning projects.

In [3]:
X = df['processed_text']
y = df['label']

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train/Test Split Complete")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("\nTraining label distribution:")
print(y_train.value_counts())
print("\nTesting label distribution:")
print(y_test.value_counts())

Train/Test Split Complete
Training samples: 57632
Testing samples: 14409

Training label distribution:
label
1    29610
0    28022
Name: count, dtype: int64

Testing label distribution:
label
1    7403
0    7006
Name: count, dtype: int64


## 2. TF-IDF Vectorization for Random Forest

TF-IDF stands for Term Frequency - Inverse Document Frequency. 
It converts text into numbers by giving higher scores to words 
that appear frequently in one article but not in many other 
articles. This makes it good for identifying words that are 
important for fake or real news classification.

I used a maximum of 50,000 features and included both single 
words and two word combinations (ngram_range=(1,2)) to capture 
more context from the articles.

In [4]:
# TF-IDF Vectorization for Random Forest
tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

# Fit on training data and transform both train and test
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF Vectorization Complete")
print("Training matrix shape:", X_train_tfidf.shape)
print("Testing matrix shape:", X_test_tfidf.shape)
print("Total features (vocabulary size):", len(tfidf_vectorizer.vocabulary_))

TF-IDF Vectorization Complete
Training matrix shape: (57632, 50000)
Testing matrix shape: (14409, 50000)
Total features (vocabulary size): 50000


## 3. Saving TF-IDF Vectorizer and Feature Matrices

The TF-IDF matrix is saved as a sparse matrix (.npz format) 
instead of a regular numpy array. This is because the full 
matrix has 57,632 rows and 50,000 features which would require 
around 21.5 GB of memory to store as a dense array.

A sparse matrix only stores the non-zero values and their 
positions. Since most articles only contain a small fraction 
of the 50,000 vocabulary words, most values in the matrix 
are zero. Saving as sparse format reduces the file size from 
21.5 GB to around 50-100 MB which is much more practical.

The vectorizer is also saved separately so it can be loaded 
in later notebooks without refitting on the data again.

In [6]:
import scipy.sparse as sp

# Save TF-IDF vectorizer
with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

# Save as sparse matrix (memory efficient)
sp.save_npz('../models/X_train_tfidf.npz', X_train_tfidf)
sp.save_npz('../models/X_test_tfidf.npz', X_test_tfidf)

# Save labels
np.save('../models/y_train.npy', y_train.values)
np.save('../models/y_test.npy', y_test.values)

print("TF-IDF vectorizer saved successfully")
print("Train and test sparse matrices saved successfully")
print("Labels saved successfully")

TF-IDF vectorizer saved successfully
Train and test sparse matrices saved successfully
Labels saved successfully


## 4. Tokenization and Padding for CNN

Unlike Random Forest which uses TF-IDF, the CNN model requires 
text to be converted into sequences of numbers. Each word in 
the vocabulary is assigned a unique number, and each article 
is converted into a sequence of those numbers.

Since articles have different lengths, padding is applied to 
make all sequences the same length. I used a maximum sequence 
length of 300 words which covers most articles in the dataset 
based on the average article length found in the EDA notebook.

In [7]:
# Tokenization for CNN
MAX_WORDS = 50000  # vocabulary size
MAX_LEN = 300      # maximum sequence length

# Create and fit tokenizer
cnn_tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
cnn_tokenizer.fit_on_texts(X_train)

print("Tokenizer fitted successfully")
print("Vocabulary size:", len(cnn_tokenizer.word_index))

Tokenizer fitted successfully
Vocabulary size: 282627


In [8]:
# Convert text to sequences
X_train_seq = cnn_tokenizer.texts_to_sequences(X_train)
X_test_seq = cnn_tokenizer.texts_to_sequences(X_test)

# Apply padding to make all sequences same length
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, 
                             padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, 
                            padding='post', truncating='post')

print("Sequences and padding complete")
print("Training padded shape:", X_train_pad.shape)
print("Testing padded shape:", X_test_pad.shape)

Sequences and padding complete
Training padded shape: (57632, 300)
Testing padded shape: (14409, 300)


## 5. Saving CNN Tokenizer and Padded Sequences

The CNN tokenizer and padded sequences are saved so they can 
be loaded directly in the model training notebook without 
repeating the tokenization process. The padded sequences are 
saved as numpy arrays since they are integer arrays and do 
not have the same memory issues as the TF-IDF float matrix.

In [9]:
# Save CNN tokenizer
with open('../models/cnn_tokenizer.pkl', 'wb') as f:
    pickle.dump(cnn_tokenizer, f)

# Save padded sequences
np.save('../models/X_train_pad.npy', X_train_pad)
np.save('../models/X_test_pad.npy', X_test_pad)

print("CNN tokenizer saved successfully")
print("Padded sequences saved successfully")
print("\nSummary of saved files:")
print("- models/tfidf_vectorizer.pkl")
print("- models/X_train_tfidf.npz")
print("- models/X_test_tfidf.npz")
print("- models/y_train.npy")
print("- models/y_test.npy")
print("- models/cnn_tokenizer.pkl")
print("- models/X_train_pad.npy")
print("- models/X_test_pad.npy")

CNN tokenizer saved successfully
Padded sequences saved successfully

Summary of saved files:
- models/tfidf_vectorizer.pkl
- models/X_train_tfidf.npz
- models/X_test_tfidf.npz
- models/y_train.npy
- models/y_test.npy
- models/cnn_tokenizer.pkl
- models/X_train_pad.npy
- models/X_test_pad.npy


## Feature Engineering Summary

In this notebook I prepared two different feature representations 
for my two models:

**For Random Forest (TF-IDF):**
- Vocabulary size: 50,000 features
- Used unigrams and bigrams (ngram_range=1,2)
- Saved as sparse matrix to save memory
- Training matrix: 57,632 articles x 50,000 features

**For CNN (Tokenized Sequences):**
- Vocabulary size: 282,627 unique words
- Maximum sequence length: 300 words
- Shorter articles padded with zeros
- Longer articles truncated to 300 words
- Training matrix: 57,632 articles x 300 sequence length

Both feature sets are saved in the models/ folder and will 
be loaded directly in Notebook 04 for model training.

## Note - Feature Engineering Update

After reviewing the group assignment proposal, Word2Vec was 
identified as the correct embedding method for the CNN model 
as stated in my individual contribution responsibilities. 
The following section implements Word2Vec using gensim library 
trained directly on the WELFake dataset. These Word2Vec embeddings 
will replace the Keras Tokenizer embeddings in the CNN model.

## 6. Word2Vec Embeddings for CNN

Word2Vec is a neural network based technique that learns word 
representations by predicting surrounding words in a sentence. 
Each word is converted into a dense vector of numbers where 
words with similar meanings have similar vectors.

I trained Word2Vec directly on the WELFake dataset instead of 
using a pretrained model. This means the word vectors will be 
specific to the language and vocabulary used in fake and real 
news articles, which should help the CNN model learn better 
representations for this classification task.

Parameters used:
- vector_size: 100 (each word represented as 100 numbers)
- window: 5 (looks at 5 surrounding words for context)
- min_count: 2 (ignores words appearing less than 2 times)
- workers: 4 (uses 4 CPU cores for faster training)

In [3]:
import pandas as pd
from gensim.models import Word2Vec
import numpy as np

# Load cleaned dataset
df_w2v = pd.read_csv('../data/cleaned_data.csv')
df_w2v = df_w2v.dropna(subset=['processed_text'])

# Prepare tokenized sentences for Word2Vec
sentences = [text.split() for text in df_w2v['processed_text']]

print("Total sentences for Word2Vec training:", len(sentences))
print("Sample sentence first 10 words:", sentences[0][:10])

Total sentences for Word2Vec training: 72041
Sample sentence first 10 words: ['law', 'enforcement', 'high', 'alert', 'following', 'threat', 'cop', 'white', 'blacklivesmatter', 'fyf']


In [4]:
# Train Word2Vec model
print("Training Word2Vec model...")

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10,
    seed=42
)

print("Word2Vec training complete")
print("Vocabulary size:", len(w2v_model.wv))
print("\nSample - vector for word 'fake' first 10 values:")
print(w2v_model.wv['fake'][:10])
print("\nMost similar words to 'fake':")
print(w2v_model.wv.most_similar('fake', topn=5))

Training Word2Vec model...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Word2Vec training complete
Vocabulary size: 171400

Sample - vector for word 'fake' first 10 values:
[-2.2105787  -1.798162   -0.15342692  3.6075351  -1.7994554  -2.154429
  0.04867032  3.1877484   1.6553013   0.6174756 ]

Most similar words to 'fake':
[('fbits', 0.6702156662940979), ('bogus', 0.6394535303115845), ('redflag', 0.6180806159973145), ('satisfiedfox', 0.6172603964805603), ('stufffox', 0.6147364377975464)]
